# Summer Analytics 2026 — Week 2 Hackathon
## Full Competition Pipeline: Top-1 Strategy

**Pipeline Phases:**
1. EDA & Data Loading
2. Missing Value Engineering
3. Advanced Feature Engineering (20 features)
4. KFold Target Encoding (leak-free)
5. Frequency Encoding
6. Model Training: CatBoost, LightGBM, XGBoost, ExtraTrees
7. Optuna Hyperparameter Tuning
8. Threshold Optimization
9. Weighted Ensemble
10. Final Submission

---
## Phase 1: Data Loading & EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print('All libraries imported successfully!')

In [ ]:
# Load datasets
train = pd.read_csv('train.csv')
public_test = pd.read_csv('public_test.csv')
private_test = pd.read_csv('private_test.csv')

print(f'Train: {train.shape}')
print(f'Public Test: {public_test.shape}')
print(f'Private Test: {private_test.shape}')
train.head()

In [ ]:
# Data types and info
train.info()

In [ ]:
# Missing values
print('=== Missing Values ===')
print(train.isnull().sum())
print(f'\nTotal missing: {train.isnull().sum().sum()}')

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x='Converted', data=train, ax=axes[0], palette='viridis')
axes[0].set_title('Target Distribution (Count)')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x()+0.35, p.get_height()+100))

train['Converted'].value_counts(normalize=True).plot.pie(
    autopct='%1.1f%%', colors=['#3498db','#e74c3c'], ax=axes[1]
)
axes[1].set_title('Target Distribution (%)')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

print(f'Converted=1: {train["Converted"].mean()*100:.2f}%')
print(f'Converted=0: {(1-train["Converted"].mean())*100:.2f}%')

In [ ]:
# Conversion rate by categorical features
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, col in enumerate(['Device_Type', 'Traffic_Source', 'City_Tier']):
    ct = pd.crosstab(train[col], train['Converted'], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i], colormap='RdYlGn')
    axes[i].set_title(f'Conversion by {col}')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45)
    axes[i].legend(['Not Converted', 'Converted'])

plt.tight_layout()
plt.show()

print('Device Type:')
print(pd.crosstab(train['Device_Type'], train['Converted'], normalize='index'))
print('\nTraffic Source:')
print(pd.crosstab(train['Traffic_Source'], train['Converted'], normalize='index'))

In [ ]:
# Numerical features boxplots
numerical_cols = ['Age', 'Income', 'Pages_Viewed', 'Products_Viewed', 
                  'Time_On_Site', 'Previous_Purchases']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for i, col in enumerate(numerical_cols):
    ax = axes[i//3][i%3]
    sns.boxplot(x='Converted', y=col, data=train, ax=ax, palette='Set2')
    ax.set_title(f'{col} by Converted')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
corrs = train[num_cols].corr()['Converted'].sort_values(ascending=False)

plt.figure(figsize=(10, 5))
corrs.drop('Converted').plot(kind='barh', color=['#2ecc71' if x > 0 else '#e74c3c' for x in corrs.drop('Converted')])
plt.title('Feature Correlations with Target (Converted)')
plt.xlabel('Correlation')
plt.tight_layout()
plt.show()

print(corrs)

In [ ]:
# Descriptive statistics
train.describe()

### EDA Key Findings
- **Target imbalance**: 30.87% Converted=1, 69.13% Converted=0
- **Strongest predictors**: `Pages_Viewed` (0.31), `Products_Viewed` (0.31), `Discount_Seen` (0.11)
- **Missing values**: Age (1,480), Income (984), Time_On_Site (1,848)
- **Outliers**: Time_On_Site max=607 (very skewed)
- **Campaign_Code**: ~9000 unique values → high cardinality, needs special encoding
- **Referral** traffic has highest conversion (38%), **Paid Ads** lowest (28%)

---
## Phase 2: Missing Value Engineering

**Strategy**: Don't just fill — create **missing indicators** first (missingness itself can be predictive), then fill with median.

In [ ]:
# Combine train + public_test for maximum training data
full_train = pd.concat([train, public_test], axis=0, ignore_index=True)
print(f'Full train (train + public): {full_train.shape}')

TARGET = 'Converted'
y_full = full_train[TARGET].values

In [ ]:
def add_missing_indicators(df):
    """Create binary flags for missing values."""
    df['Age_missing'] = df['Age'].isnull().astype(int)
    df['Income_missing'] = df['Income'].isnull().astype(int)
    df['Time_missing'] = df['Time_On_Site'].isnull().astype(int)
    df['total_missing'] = df['Age_missing'] + df['Income_missing'] + df['Time_missing']
    return df

full_train = add_missing_indicators(full_train)
private_test = add_missing_indicators(private_test)

# Fill with median (computed on train only to avoid leakage)
train_medians = {
    'Age': train['Age'].median(),
    'Income': train['Income'].median(),
    'Time_On_Site': train['Time_On_Site'].median()
}
print(f'Fill medians: {train_medians}')

for col, med in train_medians.items():
    full_train[col] = full_train[col].fillna(med)
    private_test[col] = private_test[col].fillna(med)

print(f'Missing after fill - Train: {full_train.isnull().sum().sum()}, Test: {private_test.isnull().sum().sum()}')

---
## Phase 3: Advanced Feature Engineering

We create **20 engineered features** across 5 categories:
- Engagement (ratios, intent, depth)
- Financial (loyalty, affordability)
- Discount interactions
- Cross-feature interactions
- Non-linear transforms

In [ ]:
def engineer_features(df):
    """Create 20 powerful engineered features."""
    
    # --- Engagement Features ---
    df['product_page_ratio'] = df['Products_Viewed'] / (df['Pages_Viewed'] + 1)
    df['intent_score'] = df['Products_Viewed'] * df['Time_On_Site']
    df['session_depth'] = df['Pages_Viewed'] * df['Time_On_Site']
    df['browsing_intensity'] = df['Pages_Viewed'] / (df['Time_On_Site'] + 0.1)
    df['product_intensity'] = df['Products_Viewed'] / (df['Time_On_Site'] + 0.1)
    
    # --- Financial Features ---
    df['loyalty_score'] = df['Previous_Purchases'] * df['Income']
    df['income_purchase_ratio'] = df['Income'] / (df['Previous_Purchases'] + 1)
    df['affordability'] = df['Income'] / (df['Age'] + 1)
    
    # --- Discount Features ---
    df['discount_interest'] = df['Discount_Seen'] * df['Products_Viewed']
    df['discount_intent'] = df['Discount_Seen'] * df['intent_score']
    
    # --- Interaction Features ---
    df['browser_age'] = df['Browser_Version'] * df['Age']
    df['age_income'] = df['Age'] * df['Income']
    df['pages_sq'] = df['Pages_Viewed'] ** 2
    df['products_sq'] = df['Products_Viewed'] ** 2
    
    # --- Transforms ---
    df['log_income'] = np.log1p(df['Income'])
    df['log_time'] = np.log1p(df['Time_On_Site'])
    df['time_clipped'] = df['Time_On_Site'].clip(upper=60)
    
    # --- Flags ---
    df['high_engagement'] = ((df['Pages_Viewed'] > 20) & (df['Products_Viewed'] > 15)).astype(int)
    df['returning_buyer'] = (df['Previous_Purchases'] > 0).astype(int)
    df['pages_minus_products'] = df['Pages_Viewed'] - df['Products_Viewed']
    
    # --- Composite ---
    df['engagement_composite'] = (
        df['Pages_Viewed'] * 0.3 + 
        df['Products_Viewed'] * 0.4 + 
        df['Time_On_Site'] * 0.1 + 
        df['Discount_Seen'] * 0.2
    )
    
    return df

full_train = engineer_features(full_train)
private_test = engineer_features(private_test)
print(f'Features after engineering: {full_train.shape[1]}')

---
## Phase 4: KFold Target Encoding

Target encoding replaces categorical values with the mean of the target for that category. We use **KFold** to prevent data leakage.

In [ ]:
target_encode_cols = ['Device_Type', 'Traffic_Source', 'Campaign_Code', 'Browser_Version', 'City_Tier']

def kfold_target_encode(train_df, test_df, cols, target, n_splits=5, seed=42):
    """KFold target encoding to avoid leakage."""
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    for col in cols:
        enc_col = f'{col}_te'
        train_df[enc_col] = 0.0
        
        for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(train_df, train_df[target])):
            means = train_df.iloc[tr_idx].groupby(col)[target].mean()
            train_df.loc[train_df.index[val_idx], enc_col] = train_df.iloc[val_idx][col].map(means)
        
        global_mean = train_df[target].mean()
        train_df[enc_col] = train_df[enc_col].fillna(global_mean)
        
        full_means = train_df.groupby(col)[target].mean()
        test_df[enc_col] = test_df[col].map(full_means).fillna(global_mean)
        
        print(f'  {col} -> {enc_col}')
    
    return train_df, test_df

full_train, private_test = kfold_target_encode(full_train, private_test, target_encode_cols, TARGET)

---
## Phase 5: Frequency Encoding

In [ ]:
freq_encode_cols = ['Campaign_Code', 'Browser_Version']

for col in freq_encode_cols:
    freq = full_train[col].value_counts()
    enc_col = f'{col}_freq'
    full_train[enc_col] = full_train[col].map(freq)
    private_test[enc_col] = private_test[col].map(freq).fillna(0)
    print(f'  {col} -> {enc_col} (max freq: {full_train[enc_col].max()})')

---
## Prepare Final Feature Matrix

In [ ]:
drop_cols = ['User_ID', TARGET, 'Device_Type', 'Traffic_Source']
feature_cols = [c for c in full_train.columns if c not in drop_cols]

X_full = full_train[feature_cols].values
y_full = full_train[TARGET].values
X_private = private_test[[c for c in feature_cols if c in private_test.columns]].values

print(f'Feature matrix: {X_full.shape}')
print(f'Private test: {X_private.shape}')
print(f'\nFeature names ({len(feature_cols)}):')
for i, f in enumerate(feature_cols):
    print(f'  {i+1}. {f}')

---
## Phase 6: 5-Fold Stratified Cross Validation with Multiple Models

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_model(model_fn, X, y, model_name, skf):
    """Evaluate a model with stratified k-fold CV and return OOF predictions."""
    oof_preds = np.zeros(len(y))
    fold_scores = []
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        
        model = model_fn()
        
        if model_name == 'CatBoost':
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=0)
        elif model_name == 'LightGBM':
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
        elif model_name == 'XGBoost':
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)
        else:
            model.fit(X_tr, y_tr)
        
        if hasattr(model, 'predict_proba'):
            val_proba = model.predict_proba(X_val)[:, 1]
        else:
            val_proba = model.predict(X_val)
        
        oof_preds[val_idx] = val_proba
        val_pred = (val_proba >= 0.5).astype(int)
        score = f1_score(y_val, val_pred)
        fold_scores.append(score)
        print(f'  Fold {fold+1}: F1 = {score:.5f}')
    
    mean_f1 = np.mean(fold_scores)
    std_f1 = np.std(fold_scores)
    print(f'  >> {model_name} CV F1: {mean_f1:.5f} (+/- {std_f1:.5f})')
    return oof_preds, mean_f1

In [ ]:
# CatBoost
print('--- CatBoost ---')
def catboost_fn():
    return CatBoostClassifier(
        iterations=1500, depth=6, learning_rate=0.05,
        l2_leaf_reg=3, bagging_temperature=0.8, random_strength=1.0,
        loss_function='Logloss', eval_metric='F1',
        random_seed=42, verbose=0, early_stopping_rounds=100
    )
cat_oof, cat_cv = evaluate_model(catboost_fn, X_full, y_full, 'CatBoost', skf)

In [ ]:
# LightGBM
print('--- LightGBM ---')
def lgbm_fn():
    return LGBMClassifier(
        n_estimators=1500, max_depth=6, learning_rate=0.05,
        num_leaves=40, feature_fraction=0.8, bagging_fraction=0.8,
        bagging_freq=5, min_child_samples=20,
        objective='binary', metric='binary_logloss',
        random_state=42, verbose=-1, n_jobs=-1
    )
lgb_oof, lgb_cv = evaluate_model(lgbm_fn, X_full, y_full, 'LightGBM', skf)

In [ ]:
# XGBoost
print('--- XGBoost ---')
def xgb_fn():
    return XGBClassifier(
        n_estimators=1500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42,
        verbosity=0, early_stopping_rounds=100, n_jobs=-1
    )
xgb_oof, xgb_cv = evaluate_model(xgb_fn, X_full, y_full, 'XGBoost', skf)

In [ ]:
# ExtraTrees
print('--- ExtraTrees ---')
def et_fn():
    return ExtraTreesClassifier(
        n_estimators=500, max_depth=12,
        min_samples_split=5, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    )
et_oof, et_cv = evaluate_model(et_fn, X_full, y_full, 'ExtraTrees', skf)

In [ ]:
# Compare baseline models
model_scores = {'CatBoost': cat_cv, 'LightGBM': lgb_cv, 'XGBoost': xgb_cv, 'ExtraTrees': et_cv}
plt.figure(figsize=(8, 4))
plt.barh(list(model_scores.keys()), list(model_scores.values()), color=['#2ecc71','#3498db','#e74c3c','#f39c12'])
plt.xlabel('CV F1 Score')
plt.title('Baseline Model Comparison (5-Fold CV)')
for i, (k, v) in enumerate(model_scores.items()):
    plt.text(v + 0.001, i, f'{v:.5f}', va='center')
plt.tight_layout()
plt.show()

---
## Phase 7: Optuna Hyperparameter Tuning

We tune CatBoost, LightGBM, and XGBoost with 20 Optuna trials each.

In [ ]:
# Optuna CatBoost
def optuna_catboost(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 2500),
        'depth': trial.suggest_int('depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.1, 2.0),
        'random_strength': trial.suggest_float('random_strength', 0.1, 3.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'loss_function': 'Logloss', 'eval_metric': 'F1',
        'random_seed': 42, 'verbose': 0, 'early_stopping_rounds': 100,
    }
    scores = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_full, y_full)):
        model = CatBoostClassifier(**params)
        model.fit(X_full[tr_idx], y_full[tr_idx], eval_set=(X_full[val_idx], y_full[val_idx]), verbose=0)
        val_proba = model.predict_proba(X_full[val_idx])[:, 1]
        best_f1 = max(f1_score(y_full[val_idx], (val_proba >= t).astype(int)) for t in np.arange(0.30, 0.60, 0.01))
        scores.append(best_f1)
    return np.mean(scores)

print('Tuning CatBoost (20 trials)...')
study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(optuna_catboost, n_trials=20)
print(f'Best CatBoost CV F1: {study_cat.best_value:.5f}')
print(f'Best params: {study_cat.best_params}')

In [ ]:
# Optuna LightGBM
def optuna_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2500),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 80),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'objective': 'binary', 'metric': 'binary_logloss',
        'random_state': 42, 'verbose': -1, 'n_jobs': -1,
    }
    scores = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_full, y_full)):
        model = LGBMClassifier(**params)
        model.fit(X_full[tr_idx], y_full[tr_idx], eval_set=[(X_full[val_idx], y_full[val_idx])])
        val_proba = model.predict_proba(X_full[val_idx])[:, 1]
        best_f1 = max(f1_score(y_full[val_idx], (val_proba >= t).astype(int)) for t in np.arange(0.30, 0.60, 0.01))
        scores.append(best_f1)
    return np.mean(scores)

print('Tuning LightGBM (20 trials)...')
study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(optuna_lgbm, n_trials=20)
print(f'Best LightGBM CV F1: {study_lgb.best_value:.5f}')
print(f'Best params: {study_lgb.best_params}')

In [ ]:
# Optuna XGBoost
def optuna_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2500),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 2.0),
        'eval_metric': 'logloss', 'random_state': 42,
        'verbosity': 0, 'early_stopping_rounds': 100, 'n_jobs': -1,
    }
    scores = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_full, y_full)):
        model = XGBClassifier(**params)
        model.fit(X_full[tr_idx], y_full[tr_idx], eval_set=[(X_full[val_idx], y_full[val_idx])], verbose=0)
        val_proba = model.predict_proba(X_full[val_idx])[:, 1]
        best_f1 = max(f1_score(y_full[val_idx], (val_proba >= t).astype(int)) for t in np.arange(0.30, 0.60, 0.01))
        scores.append(best_f1)
    return np.mean(scores)

print('Tuning XGBoost (20 trials)...')
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(optuna_xgb, n_trials=20)
print(f'Best XGBoost CV F1: {study_xgb.best_value:.5f}')
print(f'Best params: {study_xgb.best_params}')

---
## Phase 8-9: Retrain Tuned Models & Generate Test Predictions

In [ ]:
def retrain_tuned(study, model_class, model_name, X, y, X_test, skf, extra_params=None):
    """Retrain tuned model, collect OOF preds, and predict on test."""
    best_params = study.best_params.copy()
    if extra_params:
        best_params.update(extra_params)
    
    oof = np.zeros(len(y))
    test_preds = np.zeros(len(X_test))
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        model = model_class(**best_params)
        if model_name == 'CatBoost':
            model.fit(X[tr_idx], y[tr_idx], eval_set=(X[val_idx], y[val_idx]), verbose=0)
        elif model_name == 'LightGBM':
            model.fit(X[tr_idx], y[tr_idx], eval_set=[(X[val_idx], y[val_idx])])
        elif model_name == 'XGBoost':
            model.fit(X[tr_idx], y[tr_idx], eval_set=[(X[val_idx], y[val_idx])], verbose=0)
        
        oof[val_idx] = model.predict_proba(X[val_idx])[:, 1]
        test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits
        
        f1 = f1_score(y[val_idx], (oof[val_idx] >= 0.5).astype(int))
        print(f'  {model_name} Fold {fold+1}: F1 = {f1:.5f}')
    
    return oof, test_preds

# CatBoost tuned
print('--- Tuned CatBoost ---')
cat_extra = {'loss_function': 'Logloss', 'eval_metric': 'F1', 'random_seed': 42, 'verbose': 0, 'early_stopping_rounds': 100}
cat_oof_tuned, cat_test = retrain_tuned(study_cat, CatBoostClassifier, 'CatBoost', X_full, y_full, X_private, skf, cat_extra)

# LightGBM tuned
print('\n--- Tuned LightGBM ---')
lgb_extra = {'objective': 'binary', 'metric': 'binary_logloss', 'random_state': 42, 'verbose': -1, 'n_jobs': -1}
lgb_oof_tuned, lgb_test = retrain_tuned(study_lgb, LGBMClassifier, 'LightGBM', X_full, y_full, X_private, skf, lgb_extra)

# XGBoost tuned
print('\n--- Tuned XGBoost ---')
xgb_extra = {'eval_metric': 'logloss', 'random_state': 42, 'verbosity': 0, 'early_stopping_rounds': 100, 'n_jobs': -1}
xgb_oof_tuned, xgb_test = retrain_tuned(study_xgb, XGBClassifier, 'XGBoost', X_full, y_full, X_private, skf, xgb_extra)

---
## Phase 10: Ensemble Weight & Threshold Optimization

In [ ]:
# Search best ensemble weights and threshold jointly
best_ensemble_f1 = 0
best_weights = (0.4, 0.3, 0.3)
best_threshold = 0.5

for w1 in np.arange(0.2, 0.7, 0.05):
    for w2 in np.arange(0.1, 0.6, 0.05):
        w3 = 1.0 - w1 - w2
        if w3 < 0.05:
            continue
        ensemble_oof = w1 * cat_oof_tuned + w2 * lgb_oof_tuned + w3 * xgb_oof_tuned
        
        for t in np.arange(0.25, 0.65, 0.005):
            preds = (ensemble_oof >= t).astype(int)
            f1 = f1_score(y_full, preds)
            if f1 > best_ensemble_f1:
                best_ensemble_f1 = f1
                best_weights = (round(w1, 2), round(w2, 2), round(w3, 2))
                best_threshold = round(t, 3)

print(f'Best Ensemble Weights: CatBoost={best_weights[0]}, LightGBM={best_weights[1]}, XGBoost={best_weights[2]}')
print(f'Best Threshold: {best_threshold}')
print(f'Best Ensemble CV F1: {best_ensemble_f1:.5f}')

# Individual model thresholds
for name, oof in [('CatBoost', cat_oof_tuned), ('LightGBM', lgb_oof_tuned), ('XGBoost', xgb_oof_tuned)]:
    best_f1_s = max((f1_score(y_full, (oof >= t).astype(int)), t) for t in np.arange(0.25, 0.65, 0.005))
    print(f'  {name} alone: Best F1={best_f1_s[0]:.5f} at threshold={best_f1_s[1]:.3f}')

---
## Generate Final Submission

In [ ]:
# Ensemble test predictions
final_proba = (
    best_weights[0] * cat_test + 
    best_weights[1] * lgb_test + 
    best_weights[2] * xgb_test
)

final_preds = (final_proba >= best_threshold).astype(int)

print(f'Prediction distribution: {pd.Series(final_preds).value_counts().to_dict()}')
print(f'Positive rate: {final_preds.mean():.4f} (train was {y_full.mean():.4f})')

# Create submission
submission = pd.DataFrame({
    'User_ID': private_test['User_ID'],
    'Converted': final_preds
})

submission.to_csv('submission.csv', index=False)
print(f'\nsubmission.csv saved! Shape: {submission.shape}')
submission.head(10)

In [ ]:
# Final summary
print('=' * 60)
print('PIPELINE COMPLETE!')
print('=' * 60)
print(f'Features engineered: {len(feature_cols)}')
print(f'Models: CatBoost + LightGBM + XGBoost (Optuna-tuned)')
print(f'Ensemble weights: {best_weights}')
print(f'Optimal threshold: {best_threshold}')
print(f'CV F1 Score: {best_ensemble_f1:.5f}')